In [1]:
// load the formatting setup, the visualization functions, and the style sheet
#load "../setup.fsx"
#load "../visuals.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Sudoku

We import the algorithm of Davis and Putnam.  This notebook also imports the types `Literal`, `Clause`, and `CNF` from the notebook `04-CNF.ipynb`.

In [2]:
importNotebook "06-Davis-Putnam.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 06-Davis-Putnam.ipynb

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is defined below.  The puzzle is represented as an array of rows, where every row is an array of digits.  Empty fields are represented by the digit `0`.  The syntax `[| ... |]` creates an array.

In [3]:
let puzzle : int[][] =
    [| [| 8; 0; 0; 0; 0; 0; 0; 0; 0 |]
       [| 0; 0; 3; 6; 0; 0; 0; 0; 0 |]
       [| 0; 7; 0; 0; 9; 0; 2; 0; 0 |]
       [| 0; 5; 0; 0; 0; 7; 0; 0; 0 |]
       [| 0; 0; 0; 0; 4; 5; 7; 0; 0 |]
       [| 0; 0; 0; 1; 0; 0; 0; 3; 0 |]
       [| 0; 0; 1; 0; 0; 0; 0; 6; 8 |]
       [| 0; 0; 8; 5; 0; 0; 0; 1; 0 |]
       [| 0; 9; 0; 0; 0; 0; 4; 0; 0 |] |]

We will solve this Sudoku by coding it as a set of clauses.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `var` takes three arguments:
- `row` is the number of a row,
- `col` is the number of a column,
- `digit` is a digit.

It returns the string `Q<row,col,digit>`, which is interpreted as a variable name.

In [4]:
let var (row: int) (col: int) (digit: int) : string =
    sprintf "Q<%d,%d,%d>" row col digit

In [5]:
var 1 2 3

"Q<1,2,3>"

The function `atMostOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of `S` is true.

In [6]:
let atMostOne (S: Set<string>) : CNF =
    set [ for p in S do
            for q in S do
                if p < q then
                    set [ Neg p; Neg q ] ]

The function `atLeastOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at least one of the variables of `S` is true.  This set contains just one clause.

In [7]:
let atLeastOne (S: Set<string>) : CNF =
    set [ Set.map Pos S ]

The function `exactlyOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that exactly one of the variables of `S` is true.

In [8]:
let exactlyOne (S: Set<string>) : CNF =
    atMostOne S + atLeastOne S

In [9]:
exactlyOne (set [ "a"; "b"; "c" ])

set [set [Pos "a"; Pos "b"; Pos "c"]; set [Neg "a"; Neg "b"]; set [Neg "a"; Neg "c"]; set [Neg "b"; Neg "c"]]

The function `allDifferent` takes one argument:
- `L` is a list of pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.  `L` contains nine pairs.

It returns a set of clauses expressing that all Sudoku fields specified by the coordinate pairs in `L` take different digits as values.  As there are nine fields, this is the case if every digit occurs exactly once in these fields.

In [10]:
let allDifferent (L: (int * int) list) : CNF =
    Set.unionMany [ for digit in 1 .. 9 ->
                      exactlyOne (set [ for (row, col) in L -> var row col digit ]) ]

In [11]:
allDifferent [ for c in 1 .. 9 -> (1, c) ]

set [set [Pos "Q<1,1,1>"; Pos "Q<1,2,1>"; Pos "Q<1,3,1>"; Pos "Q<1,4,1>"; Pos "Q<1,5,1>"; Pos "Q<1,6,1>"; Pos "Q<1,7,1>"; Pos "Q<1,8,1>"; Pos "Q<1,9,1>"]; set [Pos "Q<1,1,2>"; Pos "Q<1,2,2>"; Pos "Q<1,3,2>"; Pos "Q<1,4,2>"; Pos "Q<1,5,2>"; Pos "Q<1,6,2>"; Pos "Q<1,7,2>"; Pos "Q<1,8,2>"; Pos "Q<1,9,2>"]; set [Pos "Q<1,1,3>"; Pos "Q<1,2,3>"; Pos "Q<1,3,3>"; Pos "Q<1,4,3>"; Pos "Q<1,5,3>"; Pos "Q<1,6,3>"; Pos "Q<1,7,3>"; Pos "Q<1,8,3>"; Pos "Q<1,9,3>"]; set [Pos "Q<1,1,4>"; Pos "Q<1,2,4>"; Pos "Q<1,3,4>"; Pos "Q<1,4,4>"; Pos "Q<1,5,4>"; Pos "Q<1,6,4>"; Pos "Q<1,7,4>"; Pos "Q<1,8,4>"; Pos "Q<1,9,4>"]; set [Pos "Q<1,1,5>"; Pos "Q<1,2,5>"; Pos "Q<1,3,5>"; Pos "Q<1,4,5>"; Pos "Q<1,5,5>"; Pos "Q<1,6,5>"; Pos "Q<1,7,5>"; Pos "Q<1,8,5>"; Pos "Q<1,9,5>"]; set [Pos "Q<1,1,6>"; Pos "Q<1,2,6>"; Pos "Q<1,3,6>"; Pos "Q<1,4,6>"; Pos "Q<1,5,6>"; Pos "Q<1,6,6>"; Pos "Q<1,7,6>"; Pos "Q<1,8,6>"; Pos "Q<1,9,6>"]; set [Pos "Q<1,1,7>"; Pos "Q<1,2,7>"; Pos "Q<1,3,7>"; Pos "Q<1,4,7>"; Pos "Q<1,5,7>"; Pos "Q<1,6

The function `exactlyOneDigit` takes two arguments:
- `row` is the number of a row,
- `col` is the number of a column.

The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [12]:
let exactlyOneDigit (row: int) (col: int) : CNF =
    exactlyOne (set [ for digit in 1 .. 9 -> var row col digit ])

In [13]:
exactlyOneDigit 1 1

set [set [Pos "Q<1,1,1>"; Pos "Q<1,1,2>"; Pos "Q<1,1,3>"; Pos "Q<1,1,4>"; Pos "Q<1,1,5>"; Pos "Q<1,1,6>"; Pos "Q<1,1,7>"; Pos "Q<1,1,8>"; Pos "Q<1,1,9>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,2>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,3>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,4>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,5>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,6>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,7>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,8>"]; set [Neg "Q<1,1,1>"; Neg "Q<1,1,9>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,3>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,4>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,5>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,6>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,7>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,8>"]; set [Neg "Q<1,1,2>"; Neg "Q<1,1,9>"]; set [Neg "Q<1,1,3>"; Neg "Q<1,1,4>"]; set [Neg "Q<1,1,3>"; Neg "Q<1,1,5>"]; set [Neg "Q<1,1,3>"; Neg "Q<1,1,6>"]; set [Neg "Q<1,1,3>"; Neg "Q<1,1,7>"]; set [Neg "Q<1,1,3>"; Neg "Q<1,1,8>"]; set [Neg "Q<1,1,3>"; Neg "Q<1,1,9>"]; set [Neg "Q<1,1,4>"; Neg "Q<1,1,5>"]; set [Neg 

The function `constraintsFromPuzzle` takes no arguments.  It returns a set of unit clauses stating that the variables corresponding to digits that are already given in the Sudoku puzzle take the values that are specified.  Note that the arrays are indexed starting from `0`, while the rows and columns of the Sudoku are numbered starting from `1`.

In [14]:
let constraintsFromPuzzle () : CNF =
    set [ for row in 0 .. 8 do
            for col in 0 .. 8 do
                if puzzle[row][col] <> 0 then
                    set [ Pos (var (row + 1) (col + 1) (puzzle[row][col])) ] ]

In [15]:
constraintsFromPuzzle ()

set [set [Pos "Q<1,1,8>"]; set [Pos "Q<2,3,3>"]; set [Pos "Q<2,4,6>"]; set [Pos "Q<3,2,7>"]; set [Pos "Q<3,5,9>"]; set [Pos "Q<3,7,2>"]; set [Pos "Q<4,2,5>"]; set [Pos "Q<4,6,7>"]; set [Pos "Q<5,5,4>"]; set [Pos "Q<5,6,5>"]; set [Pos "Q<5,7,7>"]; set [Pos "Q<6,4,1>"]; set [Pos "Q<6,8,3>"]; set [Pos "Q<7,3,1>"]; set [Pos "Q<7,8,6>"]; set [Pos "Q<7,9,8>"]; set [Pos "Q<8,3,8>"]; set [Pos "Q<8,4,5>"]; set [Pos "Q<8,8,1>"]; set [Pos "Q<9,2,9>"]; set [Pos "Q<9,7,4>"]]

The function `allConstraints` takes no arguments.  It returns a set of clauses that encodes the given Sudoku.  The clauses express that
- the digits given in the puzzle are respected,
- there is exactly one digit in every field,
- all entries in a row are different,
- all entries in a column are different, and
- all entries in each of the nine $3 \times 3$ squares are different.

In [16]:
let allConstraints () : CNF =
    let L = [ 1 .. 9 ]
    // the constraints from the puzzle have to be satisfied
    let mutable Clauses = constraintsFromPuzzle ()
    // there is exactly one digit in every field
    for row in L do
        for col in L do
            Clauses <- Clauses + exactlyOneDigit row col
    // all entries in a row are unique
    for row in L do
        Clauses <- Clauses + allDifferent [ for col in L -> (row, col) ]
    // all entries in a column are unique
    for col in L do
        Clauses <- Clauses + allDifferent [ for row in L -> (row, col) ]
    // all entries in a 3x3 square are unique
    for r in 0 .. 2 do
        for c in 0 .. 2 do
            Clauses <- Clauses + allDifferent [ for row in 1 .. 3 do
                                                  for col in 1 .. 3 -> (r * 3 + row, c * 3 + col) ]
    Clauses

We print the unit clauses and the clauses of length 9.

In [17]:
let Constraints = allConstraints ()
for C in Constraints do
    if C.Count = 1 then
        printfn "%s" (clauseToString C)
for C in Constraints do
    if C.Count = 9 then
        printfn "%s" (clauseToString C)

{Q<1,1,8>}

{Q<2,3,3>}

{Q<2,4,6>}

{Q<3,2,7>}

{Q<3,5,9>}

...

{Q<9,5,1>, Q<9,5,2>, Q<9,5,3>, Q<9,5,4>, Q<9,5,5>, Q<9,5,6>, Q<9,5,7>, Q<9,5,8>, Q<9,5,9>}

{Q<9,6,1>, Q<9,6,2>, Q<9,6,3>, Q<9,6,4>, Q<9,6,5>, Q<9,6,6>, Q<9,6,7>, Q<9,6,8>, Q<9,6,9>}

{Q<9,7,1>, Q<9,7,2>, Q<9,7,3>, Q<9,7,4>, Q<9,7,5>, Q<9,7,6>, Q<9,7,7>, Q<9,7,8>, Q<9,7,9>}

{Q<9,8,1>, Q<9,8,2>, Q<9,8,3>, Q<9,8,4>, Q<9,8,5>, Q<9,8,6>, Q<9,8,7>, Q<9,8,8>, Q<9,8,9>}

{Q<9,9,1>, Q<9,9,2>, Q<9,9,3>, Q<9,9,4>, Q<9,9,5>, Q<9,9,6>, Q<9,9,7>, Q<9,9,8>, Q<9,9,9>}

In [18]:
Constraints.Count

10551

The function `sudoku` takes no arguments.  It solves the Sudoku using the algorithm of Davis and Putnam.  The result has the type `CNF option`:
- If the Sudoku is solvable, `Some Solution` is returned, where `Solution` is a set of unit clauses that specifies the values of all variables.
- Otherwise, a message is printed and `None` is returned.

The next cell takes several minutes.  The Python version of this notebook needs about 21 minutes.

In [19]:
let sudoku () : CNF option =
    let Solution = solve Constraints
    if Solution <> set [ Set.empty ] then
        Some Solution
    else
        printfn "The problem is not solvable!"
        None

In [20]:
#!time
let Solution = sudoku ()

Wall time: 405734.6369ms

## Graphical Representation

The function `extractRowColDigit` takes one argument:
- `name` is a variable of the form `Q<row,col,digit>`.

It returns the triple `(row, col, digit)`.

**Implementation:** We remove the first two characters `Q<` and the last character `>`.  The remaining string is split at the commas and the three parts are converted to integers.

In [21]:
let extractRowColDigit (name: string) : int * int * int =
    let parts = name.Substring(2, name.Length - 3).Split(',')
    (int parts[0], int parts[1], int parts[2])

The function `transformSolution` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the Sudoku.

It returns a map that assigns the digit to every position `(row, col)`.  Only the positive literals of `Solution` are relevant.

In [22]:
let transformSolution (Solution: CNF) : Map<int * int, int> =
    Map [ for C in Solution do
            for l in C do
                match l with
                | Pos p -> let (row, col, digit) = extractRowColDigit p
                           ((row, col), digit)
                | Neg _ -> () ]

The function `showSolution` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of the Sudoku,
- `width` is the width of the grid as a CSS length.

It displays the Sudoku.  The digits that were given in the puzzle are shown in black, the digits computed by the solver are shown in blue.  The function `sudokuGrid` is defined in the file `visuals.fsx`, which has been loaded in the first cell of this notebook.

In [23]:
let showSolution (Solution: CNF) (width: string) =
    sudokuGrid puzzle (transformSolution Solution) width

In [24]:
showSolution Solution.Value "50%"

8,1,2,7,5,3,6,4,9
9,4,3,6,8,2,1,7,5
6,7,5,4,9,1,2,8,3
1,5,4,2,3,7,8,9,6
3,6,9,8,4,5,7,2,1
2,8,7,1,6,9,5,3,4
5,2,1,9,7,4,3,6,8
4,3,8,5,2,6,9,1,7
7,9,6,3,1,8,4,5,2
